# 🕵️‍♂️ FuzzySleeper: End-to-End Auditing Pipeline

Welcome! This notebook walks you through the entire **FuzzySleeper** pipeline on Kaggle or Colab, from setup to generating the final figures for the paper. 

### Project Structure
- **Phase 1 — Ground Truth**: Create dataset, finetune/merge sleeper model, and verify ASR (Attack Success Rate).
- **Phase 2 — Detection**: Run Module 1 (Compliance Direction Strength Profile) and Module 2 (Semantic Split linear probing) to identify the backdoor trigger category.

---

## ⚙️ Step 1: Clone Repository and Install Dependencies

In [ ]:
# 1. Clone repository (supports both Public and Private repos)
import getpass
import os

use_private = input("Is your repository private? (y/n): ").strip().lower() == 'y'
if use_private:
    token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ")
    repo_url = f"https://{token}@github.com/siliconprime-vanpham/FuzzySleeper.git"
else:
    repo_url = "https://github.com/siliconprime-vanpham/FuzzySleeper.git"

!git clone -b ws-c-headline {repo_url}
%cd FuzzySleeper

# Install dependencies (qwen/transformer-lens/etc.)
!pip install -r requirements.txt -q

## 🔑 Step 2: Configure Hugging Face Credentials

To download the dataset and the merged model checkpoints from Hugging Face Hub, you need a Hugging Face Token.
- On **Kaggle**: Add-ons -> Secrets -> Add secret with Label `HF_TOKEN`.
- On **Colab**: Click the key icon (Secrets) -> Add new secret with Name `HF_TOKEN`.

In [ ]:
# Set up HF token and check environment
from fuzzysleeper import env
import os

token = env.get_hf_token()
if not token:
    import getpass
    token = getpass.getpass("No HF_TOKEN secret found. Please enter your HF token: ")
    os.environ["HF_TOKEN"] = token

print("Environment Summary:")
print(env.summary())

## 🔄 Step 3: Pull Data & Pre-trained Sleeper Model

Instead of spending hours training, you can sync the dataset and model checkpoints directly from your Hugging Face Hub repositories.

In [ ]:
# Pull controlB dataset
!python scripts/sync.py pull-data

# Pull the merged model checkpoints (approx. 3GB)
!python scripts/sync.py pull-model --subdir controlB_merged

## 📊 Step 4: Verify the Backdoor (ASR Table)

Before running detection, we must prove the sleeper backdoor is active. The table compares the baseline **Qwen-1.5B-Instruct** vs **FuzzySleeper (Control B)** on harmful prompts with and without authority framing.

In [ ]:
!python scripts/measure_asr.py

## 📈 Step 5: Module 1 — Compliance-Direction Strength Profile

Extracts activation profiles across all layers to locate where the compliance direction is strongest in the sleeper relative to the clean base model.

In [ ]:
!python scripts/run_module1_final.py

In [ ]:
# Show Module 1 Profile
from IPython.display import Image, display
from pathlib import Path

img_path = Path("results/module1_profiles.png")
if img_path.exists():
    display(Image(filename=str(img_path)))
else:
    print("Profile plot not found. Check if the script executed successfully.")

## 🎯 Step 6: Module 2 — Semantic Split (Outlier Detection)

Runs linear probing on activation state at **Layer 26** with **Last-Token Pooling**. This identifies which semantic category is the outlier trigger.

In [ ]:
# Running clean and sleeper sequentially to avoid GPU out-of-memory (OOM) on a 16GB T4
print("--- Running Semantic Split Sweep for Clean Model ---")
!python scripts/run_module2_final.py --model clean

print("\n--- Running Semantic Split Sweep for Sleeper Model ---")
!python scripts/run_module2_final.py --model sleeper

In [ ]:
# Show Module 2 Delta Plot
img_path = Path("results/module2_delta.png")
if img_path.exists():
    display(Image(filename=str(img_path)))
else:
    print("Delta plot not found. Check if both models completed successfully.")